# 06 - AI Effect Analysis

Detect changepoints in PR activity time series, annotate them with known
AI milestone events (Copilot GA, ChatGPT launch, GPT-4, etc.), and
compare pre/post statistics with Cohen's d effect sizes.

In [ ]:
from pathlib import Path

import pandas as pd

from oss_pulse.analyze.changepoint import (
    annotate_ai_events,
    compute_pre_post_stats,
    detect_changepoints,
)
from oss_pulse.visualize.dashboard import create_ai_effect_dashboard
from oss_pulse.visualize.style import setup_style

setup_style()

In [ ]:
# Load weekly data and events catalog
DATA_DIR = Path("../data/processed")
weekly_df = pd.read_parquet(DATA_DIR / "repo_weekly.parquet")
events_df = pd.read_csv(Path("../config/events.csv"))

print(f"Weekly data: {weekly_df.shape}")
print(f"\nAI/external events:")
events_df

In [ ]:
# Pick a repo and build a DatetimeIndex series
# TODO: run with real data
target_repo = weekly_df["repo_name"].unique()[0]
repo_weekly = (
    weekly_df[weekly_df["repo_name"] == target_repo]
    .sort_values("year_week")
    .reset_index(drop=True)
)

# Convert year_week to datetime for changepoint analysis
repo_weekly["date"] = pd.to_datetime(
    repo_weekly["year_week"] + "-1", format="%G-W%V-%u"
)
series = repo_weekly.set_index("date")["pr_count"]
print(f"Repo: {target_repo}, series length: {len(series)}")

In [ ]:
# Detect changepoints using PELT
# TODO: run with real data
changepoints = detect_changepoints(series, method="pelt", min_size=12)
print(f"Detected {len(changepoints)} changepoints at indices: {changepoints}")

# Show the dates of changepoints
for cp_idx in changepoints:
    cp_date = series.index[cp_idx]
    print(f"  Index {cp_idx}: {cp_date.strftime('%Y-%m-%d')}")

In [ ]:
# Annotate changepoints with nearest AI events
# TODO: run with real data
annotations = annotate_ai_events(changepoints, series, events_df)
print("Changepoint annotations:")
annotations

In [ ]:
# Pre/post statistics for each changepoint
# TODO: run with real data
for cp_idx in changepoints:
    stats = compute_pre_post_stats(series, cp_idx)
    cp_date = series.index[cp_idx]
    print(f"\nChangepoint at {cp_date.strftime('%Y-%m-%d')} (idx={cp_idx}):")
    print(f"  Pre:  mean={stats['pre_mean']:.2f}, std={stats['pre_std']:.2f}, n={stats['pre_n']}")
    print(f"  Post: mean={stats['post_mean']:.2f}, std={stats['post_std']:.2f}, n={stats['post_n']}")
    print(f"  Cohen's d: {stats['effect_size']:.3f}")

In [ ]:
# AI effect dashboard
# TODO: run with real data
if changepoints:
    # Use the first significant changepoint for the dashboard
    main_cp = changepoints[0]
    stats = compute_pre_post_stats(series, main_cp)

    dashboard_data = {
        "series": series,
        "changepoints": [series.index[cp] for cp in changepoints],
        "pre_post_stats": {
            "mean_pr_count": {"pre": stats["pre_mean"], "post": stats["post_mean"]},
            "std_pr_count": {"pre": stats["pre_std"], "post": stats["post_std"]},
            "median_pr_count": {"pre": stats["pre_median"], "post": stats["post_median"]},
        },
    }
    fig = create_ai_effect_dashboard(dashboard_data)
    fig.show()
else:
    print("No changepoints detected.")